**1. SETUP:**
In this section, I define the file paths, countries, and thresholds
as configuration variables.

The run header displays the countries to be loaded and the start time
using f-strings.

- I used clear headings for each section.
- I arranged the code cells in the correct order.
- I will use Restart and Run All to check the notebook from the beginning.
- I defined all required variables in the notebook to avoid hidden state.

In [ ]:
import pandas as pd
from datetime import datetime

folder = "/content/"

file_paths = {"India": folder + "INvideos.csv", "USA": folder + "USvideos.csv",
              "UK": folder + "GBvideos.csv"}

category_path = folder + "categories.csv"
countries = ["India", "USA", "UK"]
viral_views, steady_views, low_views = 1000000, 100000, 10000
high_like_ratio, divisive_like_ratio, max_rows = 0.80, 0.60, None
start_time = datetime.now()
print(f"Countries: {countries} | Run started: {start_time}")

Countries: ['India', 'USA', 'UK'] | Run started: 2026-10-27 14:10:01.314066


**2. LOADING FUNCTIONS:**
The `load_country()` function reads a CSV file using pandas.

It checks whether the required columns are present, adds a country
column, and returns the data as a list of dictionaries.

The `max_rows` parameter has a default value of `None`, which means
all rows will be loaded.

A dictionary called `category_names` stores each category ID and
its corresponding category name.

In [ ]:
#1. Function:

required = ["video_id", "title", "channel_title", "category_id", "trending_date",
            "publish_time", "views", "likes", "dislikes", "comment_count",
            "comments_disabled", "ratings_disabled", "video_error_or_removed"]

def load_country(path, country, max_rows=None):
    df = pd.read_csv(path, nrows=max_rows)
    for column in required:
        if column not in df.columns:
            raise ValueError(f"{country}: missing {column}")
    df["country"] = country
    print(f"{country}: {len(df)} records loaded")
    return df.to_dict("records")

In [ ]:
#2.Load all countries:

records = []

for country in countries:
    rows = load_country(file_paths[country], country)
    records.extend(rows)

print(f"Total records: {len(records)}")

India: 15075 records loaded
USA: 16080 records loaded
UK: 15075 records loaded
Total records: 46230


In [ ]:
#3.Check category columns:

categories_df = pd.read_csv(category_path)

print(categories_df.columns.tolist())
categories_df.head()

['category_id', 'category_name']


,category_id,category_name
0,1,Film & Animation
1,2,Autos & Vehicles
2,10,Music
3,15,Pets & Animals
4,17,Sports


In [ ]:
#4.Category dictionary:

category_names = {}

for _, row in categories_df.iterrows():
    category_names[row["category_id"]] = row["category_name"]

print(category_names)

{1: 'Film & Animation', 2: 'Autos & Vehicles', 10: 'Music', 15: 'Pets & Animals', 17: 'Sports', 19: 'Travel & Events', 20: 'Gaming', 22: 'People & Blogs', 23: 'Comedy', 24: 'Entertainment', 25: 'News & Politics', 26: 'Howto & Style', 27: 'Education', 28: 'Science & Technology', 29: 'Nonprofits & Activism', 30: 'Movies', 43: 'Shows'}


**3.Type Conversion and Cleaning**

I convert numeric fields to integers, true/false fields to booleans,
and date fields to date or datetime values.

Invalid or blank values are replaced with None. I keep the records
because their other fields may contain useful data. I also count and
print the failed conversions for each field.

In [ ]:
#1.Fields and failure count:

numbers = ["views", "likes", "dislikes", "comment_count"]
flags = ["comments_disabled", "ratings_disabled", "video_error_or_removed"]
dates = ["trending_date", "publish_time"]

failures = {}

for field in numbers + flags + dates:
    failures[field] = 0
print(failures[field])

0


In [ ]:
#2. Number cleaning function:

def clean_number(value):
    number = float(value)

    if number < 0 or not number.is_integer():
        raise ValueError("Invalid number")

    return int(number)


In [ ]:
#3. Numbers change into integers:

for row in records:
    for field in numbers:
        try:
            row[field] = clean_number(row[field])
        except (ValueError, TypeError, OverflowError):
            row[field] = None
            failures[field] += 1

In [ ]:
#4. Boolean cleaning function:

def clean_boolean(value):
    text = str(value).strip().lower()

    if text == "true":
        return True
    elif text == "false":
        return False
    else:
        raise ValueError("Invalid boolean")

In [ ]:
#5. Change True or False:

for row in records:
    for field in flags:
        try:
            row[field] = clean_boolean(row[field])
        except (ValueError, TypeError):
            row[field] = None
            failures[field] += 1

In [ ]:
#6. Change trending Date:

for row in records:
    try:
        row["trending_date"] = datetime.strptime(
            str(row["trending_date"]), "%y.%d.%m"
        ).date()
    except (ValueError, TypeError):
        row["trending_date"] = None
        failures["trending_date"] += 1

In [ ]:
#7. Change Publish Time and Print Failure Counts:

for row in records:
    try:
        value = str(row["publish_time"]).replace("Z", "+00:00")
        row["publish_time"] = datetime.fromisoformat(value)
    except (ValueError, TypeError):
        row["publish_time"] = None
        failures["publish_time"] += 1

for field, count in failures.items():
    print(f"{field}: {count} failed")

views: 0 failed
likes: 12 failed
dislikes: 0 failed
comment_count: 36 failed
comments_disabled: 0 failed
ratings_disabled: 0 failed
video_error_or_removed: 0 failed
trending_date: 0 failed
publish_time: 0 failed


**4. Core Data Structures**

- A list stores all cleaned records. It is mutable.
- A dictionary stores each record using field names. It is mutable.
- A tuple stores selected fields as a fixed snapshot. It is immutable.
- A set stores unique video IDs and compares countries. It is mutable.
- A nested dictionary stores record counts by country and category.
  It is mutable

In [ ]:
#1. List and Tuples:

video_tuples = []

for row in records:
    video_tuples.append(
        (row["video_id"], row["trending_date"], row["views"], row["likes"])
    )

print("First 5 video tuples:")
print(video_tuples[:5])

First 5 video tuples:
[('UcYiQY22yfG', datetime.date(2017, 11, 14), 217915, 14918), ('EutcTEIciae', datetime.date(2017, 11, 14), 115014, 2739), ('rvuZ5CsBpg8', datetime.date(2017, 11, 14), 51028, 1786), ('2yJGS3uxgMP', datetime.date(2017, 11, 14), 143370, 5224), ('W5pdmLZu7GK', datetime.date(2017, 11, 14), 390220, 15079)]


In [ ]:
#2. Unique Videos IDs for each countries:

country_ids = {"India": set(), "USA": set(), "UK": set()}

for row in records:
    if pd.notna(row["video_id"]) and str(row["video_id"]).strip():
        country_ids[row["country"]].add(row["video_id"])

for country in countries:
    print(f"{country}: {len(country_ids[country])} unique videos")

India: 5878 unique videos
USA: 5068 unique videos
UK: 4113 unique videos


In [ ]:
#3. Trending videos in one and more countries;

india = country_ids["India"]
usa = country_ids["USA"]
uk = country_ids["UK"]

common = (india & usa) | (india & uk) | (usa & uk)

print(f"Videos trending in more than one country: {len(common)}")

Videos trending in more than one country: 340


In [ ]:
#4. Trending videos in singlr country:

india_only = india - usa - uk
usa_only = usa - india - uk
uk_only = uk - india - usa

print(f"India only: {len(india_only)}")
print(f"USA only: {len(usa_only)}")
print(f"UK only: {len(uk_only)}")

India only: 5652
USA only: 4834
UK only: 3873


In [ ]:
#5. Channel appearences count:

channel_counts = {}

for row in records:
    channel = row["channel_title"]
    if pd.notna(channel) and str(channel).strip():
        channel_counts[channel] = channel_counts.get(channel, 0) + 1

print(f"Total channels: {len(channel_counts)}")

Total channels: 438


In [ ]:
#6. Top 15 channels using Slicing:

sorted_channels = sorted(
    channel_counts.items(),
    key=lambda item: item[1],
    reverse=True
)

for channel, count in sorted_channels[:15]:
    print(f"{channel}: {count}")

QuietChallenge75: 2464
OfficialReview41: 1586
UltimateUnboxing65: 1433
InsaneReaction46: 1233
Brand NewChallenge45: 1175
CrazyTrailer91: 1018
FinalTrailer17: 911
UltimateSpeech20: 770
Brand NewSecret51: 684
WildHighlights7: 682
SurprisingHighlights41: 650
FinalBreakdown11: 643
CrazyRecipe67: 603
QuietInterview9: 570
UltimateVlog96: 565


In [ ]:
#7. Country → Category → Record count:

summary = {}

for row in records:
    country = row["country"]
    category = category_names.get(row["category_id"], "Unknown")
    if country not in summary:
        summary[country] = {}
    summary[country][category] = summary[country].get(category, 0) + 1

In [ ]:
#8.Print singlr part in summary

for country, categories in summary.items():
    print(f"\n{country}:")

    for category, count in list(categories.items())[:5]:
        print(f"{category}: {count} records")


India:
Music: 2298 records
Film & Animation: 770 records
Entertainment: 5117 records
Comedy: 966 records
People & Blogs: 1098 records

USA:
People & Blogs: 1423 records
News & Politics: 755 records
Music: 3052 records
Education: 561 records
Entertainment: 4371 records

UK:
Science & Technology: 470 records
Howto & Style: 588 records
Comedy: 941 records
Entertainment: 2844 records
Music: 5013 records


**5. Variable Scope**

A global variable is defined outside a function and can be read
inside the function.

A local variable is defined inside a function and cannot be
accessed directly outside the function.

In [ ]:
project_name = "YouTube Analysis"

def show_scope():
    message = "Hello from the function"
    print("Inside function:")
    print("Global variable:", project_name)
    print("Local variable:", message)

show_scope()

Inside function:
Global variable: YouTube Analysis
Local variable: Hello from the function


In [ ]:
print("Outside function:")
print("Global variable:", project_name)

try:
    print("Local variable:", message)
except NameError:
    print("Local variable cannot be accessed outside the function")

Outside function:
Global variable: YouTube Analysis
Local variable cannot be accessed outside the function


**6. Loops over Strings and Sequences**

The title_features() function counts uppercase words, exclamation
marks, question marks, and the total number of words in a title.

A for loop checks the words, and range() with index access checks
the characters. A while loop searches for the first video above
the view threshold, using continue and break.

In [ ]:
#1. title_features() function:

def title_features(title):
    words = title.split()
    uppercase = 0
    for word in words:
        if word.strip("!?.,").isupper():
            uppercase += 1
    exclamations, questions = count_marks(title)
    return uppercase, exclamations, questions, len(words)

In [ ]:
#2. Count the Characters through range():

def count_marks(title):
    exclamations = 0
    questions = 0
    for i in range(len(title)):
        if title[i] == "!":
            exclamations += 1
        elif title[i] == "?":
            questions += 1
    return exclamations, questions

In [ ]:
#3. Check the Title:

title = "WOW! Is this REAL?"
uppercase, exclamations, questions, word_count = title_features(title)
print(f"Title: {title}")
print(f"Uppercase words: {uppercase}")
print(f"Exclamation marks: {exclamations}")
print(f"Question marks: {questions}")
print(f"Word count: {word_count}")

Title: WOW! Is this REAL?
Uppercase words: 2
Exclamation marks: 1
Question marks: 1
Word count: 4


In [ ]:
#4. while loop, continue, break:

index = 0
while index < len(records):
    row = records[index]
    index += 1
    if row["views"] is None:
        continue
    if row["views"] > viral_views:
        print(f"Title: {row['title']} | Views: {row['views']}")
        break

Title: Official Pasta Finale | Views: 1029375


** 7. Rule-Based Classification**

Videos are classified using views and like ratio:

- Viral Hit: views >= 1,000,000 and ratio >= 0.80.
- Divisive: ratio < 0.60.
- Low Engagement: views < 10,000.
- Steady Performer: views >= 100,000.
- Unrated: missing values, zero views, disabled ratings,
  zero ratings, or no matching rule.

Rules are checked in this order. A missing comments-disabled flag
also gives Unrated; disabled comments alone do not change the label.

Example: 2,000,000 views, 900 likes and 100 dislikes give a ratio
of 0.90. With ratings enabled, the label is Viral Hit.

In [ ]:
#1.Classification function:

def classify_video(row):
    v, l, d = row["views"], row["likes"], row["dislikes"]
    flags = (row["ratings_disabled"], row["comments_disabled"])
    if None in (v, l, d, *flags) or v == 0 or flags[0]:
        return "Unrated"
    if l + d == 0:
        return "Unrated"
    return classify_rated(v, l, d)

In [ ]:
def classify_rated(views, likes, dislikes):
    ratio = likes / (likes + dislikes)
    if views >= viral_views:
        if ratio >= high_like_ratio:
            return "Viral Hit"
    return other_label(views, ratio)

In [ ]:
#2.Lable Function:

def other_label(views, ratio):
    if ratio < divisive_like_ratio:
        return "Divisive"
    elif views < low_views:
        return "Low Engagement"
    elif views >= steady_views:
        return "Steady Performer"
    else:
        return "Unrated"



In [ ]:
#3. Count the all records by giving lables for all:

label_counts = {}

for row in records:
    label = classify_video(row)
    row["label"] = label
    label_counts[label] = label_counts.get(label, 0) + 1

print(label_counts)

for label, count in label_counts.items():
    print(f"{label}: {count}")

{'Steady Performer': 27271, 'Unrated': 9027, 'Viral Hit': 9497, 'Divisive': 251, 'Low Engagement': 184}
Steady Performer: 27271
Unrated: 9027
Viral Hit: 9497
Divisive: 251
Low Engagement: 184


**8. Refactoring**

I used the `clean_number()` function from Question 3 to clean views,
likes, dislikes, and comment counts.

This avoids repeating the same code. It also makes the cleaning rules
easier to update.

**9. Readme:**

## Files and Countries
- India: INvideos.csv
- USA: USvideos.csv
- UK: GBvideos.csv
- categories.csv: category names

## How to Run
1. Open the notebook in Google Colab.
2. Upload the four CSV files to `/content/`.
3. Pandas is usually installed in Colab. If needed, run `!pip install pandas`.
4. Select Runtime → Restart session and run all.

## Cleaning Rules
- Convert counts to non-negative integers, flags to booleans, and dates
  to date or datetime values.
- Replace blank or invalid values with `None` and count failures.
- Keep the records because their other fields may be useful.

## Classification Thresholds
Rules are checked in this order:
- Viral Hit: views >= 1,000,000 and like ratio >= 0.80.
- Divisive: like ratio < 0.60.
- Low Engagement: views < 10,000.
- Steady Performer: views >= 100,000.
- Unrated: missing required values, zero views, disabled ratings,
  zero total ratings, or no other rule matches.

Like ratio = likes / (likes + dislikes).
A missing comments-disabled flag also results in Unrated.
